In [1]:
import numpy as np

np.random.seed(0)

# Generate data: x ~ N(mu_true, sigma2_true)
mu_true = 2.0
sigma2_true = 1.5
sigma_true = np.sqrt(sigma2_true)

Nmax = 1000
x = np.random.normal(mu_true, sigma_true, size=Nmax)

# Part A) Infer mu with known sigma^2  (Gaussian-Gaussian conjugacy)
# Prior:  mu ~ N(m0, s0^2)
# Likelihood: x_n ~ N(mu, sigma^2) with sigma^2 known
#
# Posterior: mu | x ~ N(mN, sN^2)
# sN^2 = 1 / ( 1/s0^2 + N/sigma^2 )
# mN   = sN^2 * ( m0/s0^2 + (N/sigma^2)*xbar )
#
# MLE: mu_ML = xbar
m0 = 0.0
s02 = 10.0**2  # weak prior

def posterior_mu(xN, sigma2, m0, s02):
    N = len(xN)
    xbar = xN.mean()
    sN2 = 1.0 / (1.0/s02 + N/sigma2)
    mN = sN2 * (m0/s02 + (N/sigma2)*xbar)
    return mN, sN2, xbar

# Part B) Infer sigma^2 with known mu  (Gamma conjugate on precision)
# Bishop (pp. 97–99): using precision beta = 1/sigma^2
# Prior:  p(beta) = Gam(beta | a0, b0)  (shape a0, rate b0)
# Likelihood: x_n ~ N(mu, 1/beta) with mu known
#
# Posterior: beta | x ~ Gam(beta | aN, bN)
# aN = a0 + N/2
# bN = b0 + 0.5 * sum_n (x_n - mu)^2

# E[beta] = aN / bN
# E[sigma^2] = E[1/beta] = bN / (aN - 1)   (requires aN > 1)
# ML: sigma2_ML = (1/N) * sum (x_n - mu)^2   (when mu known)
a0 = 1.0
b0 = 1.0

def posterior_sigma2_via_precision(xN, mu_known, a0, b0):
    N = len(xN)
    S = np.sum((xN - mu_known)**2)
    aN = a0 + 0.5*N
    bN = b0 + 0.5*S
    beta_mean = aN / bN
    sigma2_mean = bN / (aN - 1.0) if aN > 1.0 else np.nan
    sigma2_ML = S / N
    return aN, bN, beta_mean, sigma2_mean, sigma2_ML

# Run for different N and compare Bayesian estimates to ML
Ns = [5, 20, 100, 1000]

print("A) Infer mu with known sigma^2")
for N in Ns:
    xN = x[:N]
    mN, sN2, mu_ML = posterior_mu(xN, sigma2_true, m0, s02)
    print(f"N={N:4d}  mu_ML={mu_ML: .4f}   posterior mean mN={mN: .4f}   posterior std={np.sqrt(sN2): .4f}")

print("\nB) Infer sigma^2 with known mu (Gamma prior on precision beta=1/sigma^2)")
mu_known = mu_true  # assumed known in this part (as the activity states)
for N in Ns:
    xN = x[:N]
    aN, bN, beta_mean, sigma2_mean, sigma2_ML = posterior_sigma2_via_precision(xN, mu_known, a0, b0)
    print(f"N={N:4d}  sigma2_ML={sigma2_ML: .4f}   E[sigma^2]={sigma2_mean: .4f}   E[beta]={beta_mean: .4f}   (aN={aN:.2f}, bN={bN:.2f})")


A) Infer mu with known sigma^2
N=   5  mu_ML= 3.7762   posterior mean mN= 3.7649   posterior std= 0.5469
N=  20  mu_ML= 2.6973   posterior mean mN= 2.6953   posterior std= 0.2738
N= 100  mu_ML= 2.0732   posterior mean mN= 2.0729   posterior std= 0.1225
N=1000  mu_ML= 1.9446   posterior mean mN= 1.9445   posterior std= 0.0387

B) Infer sigma^2 with known mu (Gamma prior on precision beta=1/sigma^2)
N=   5  sigma2_ML= 3.8218   E[sigma^2]= 4.2218   E[beta]= 0.3316   (aN=3.50, bN=10.55)
N=  20  sigma2_ML= 1.5704   E[sigma^2]= 1.6704   E[beta]= 0.6585   (aN=11.00, bN=16.70)
N= 100  sigma2_ML= 1.5291   E[sigma^2]= 1.5491   E[beta]= 0.6584   (aN=51.00, bN=77.46)
N=1000  sigma2_ML= 1.4644   E[sigma^2]= 1.4664   E[beta]= 0.6833   (aN=501.00, bN=733.21)


### Conjugate priors and iterative Bayesian inference (Bishop Ch. 2, pp. 97–99)

### Report note: Conjugate priors and iterative Bayesian inference (Bishop Ch. 2, pp. 97–99)

A **conjugate prior** is a prior distribution chosen so that, after combining it with the likelihood, the **posterior has the same functional form** as the prior. This is useful because Bayesian updating becomes a set of simple parameter updates (no difficult integrals).

#### 1) Unknown mean $\mu$ with known variance $\sigma^2$ (Gaussian–Gaussian conjugacy)

Assume i.i.d. data with likelihood

$$
p(x_n \mid \mu)=\mathcal{N}(x_n \mid \mu,\sigma^2), \qquad \sigma^2 \ \text{known}.
$$

Choose a Gaussian prior

$$
p(\mu)=\mathcal{N}(\mu \mid m_0,s_0^2).
$$

Then the posterior is also Gaussian

$$
p(\mu \mid x_{1:N})=\mathcal{N}(\mu \mid m_N,s_N^2),
$$

with closed-form updates

$$
s_N^2=\left(\frac{1}{s_0^2}+\frac{N}{\sigma^2}\right)^{-1},
\qquad
m_N=s_N^2\left(\frac{m_0}{s_0^2}+\frac{N}{\sigma^2}\,\bar{x}\right),
$$

where

$$
\bar{x}=\frac{1}{N}\sum_{n=1}^N x_n.
$$

The maximum-likelihood estimate is

$$
\mu_{\mathrm{ML}}=\bar{x}.
$$

The posterior mean $m_N$ is a precision-weighted compromise between prior belief and data, and as N increases, we get $\mu_N$ to $\mu_{\mathrm{ML}}$). The posterior standard deviation shrinks like $1/\sqrt{N}$.

#### 2) Unknown variance $\sigma^2$ with known mean $\mu$ (Gamma prior on precision)

Following Bishop, define the precision $\beta = 1/\sigma^2$. With known $\mu$,

$$
p(x_n \mid \beta)=\mathcal{N}(x_n \mid \mu,\beta^{-1}).
$$

Use a Gamma prior on \(\beta\) (rate parameterization):

$$
p(\beta)=\mathrm{Gam}(\beta \mid a_0,b_0).
$$

Then the posterior is also Gamma:

$$
p(\beta \mid x_{1:N})=\mathrm{Gam}(\beta \mid a_N,b_N),
$$

with updates

$$
a_N=a_0+\frac{N}{2},
\qquad
b_N=b_0+\frac{1}{2}\sum_{n=1}^N (x_n-\mu)^2.
$$

With known \(\mu\), the ML estimate of the variance is

$$
\sigma^2_{\mathrm{ML}}=\frac{1}{N}\sum_{n=1}^N (x_n-\mu)^2.
$$

#### Iterative (sequential) Bayesian algorithms

Conjugacy makes Bayesian inference naturally **iterative**: after observing $N-1$ points, the posterior parameters become the prior parameters for the N-th update. Practically, you maintain only **sufficient statistics** instead of storing all data.

For unknown $\mu$ (known $\sigma^2$), you update N and $\sum x_n$ (or $\bar{x}$), then update $m_N$ and $s_N^2$.

For unknown $\sigma^2$ (known $\mu$), you update the running sum of squares $\sum (x_n-\mu)^2$, then update $a_N$ and $b_N$.

Because the posterior stays in the same family (Gaussian or Gamma), these updates are fast, stable, and work online with constant memory.
